### Preprocessing Steps for Tunisian Telecom Reviews

In this notebook, we performed several preprocessing steps to clean and prepare the raw review data for further analysis. Below is a summary of the steps:

---

#### 1. **Loading Raw Data**
- We loaded the raw dataset from the file located at `../data/01_raw/tunisian_telco_reviews_raw.csv`.
- The dataset contains reviews from Tunisian telecom customers, including mixed-language content (French, Arabic, and Derja).

---

#### 2. **Handling Missing Data**
- We dropped rows where the `content` column (review text) was `NaN` to ensure we only work with valid reviews.

---

#### 3. **Filtering Non-Derja Digits**
- Reviews containing non-Derja digits (`0`, `1`, `4`, `6`) were removed using a regular expression. This step ensures that the data aligns with the transliteration rules for Derja.

---

#### 4. **Text Cleaning**
- We initialized a `TunisianTextCleaner` object to handle the unique challenges of mixed-language reviews.
- The cleaner was applied to the `content` column using the `progress_apply` method, which provided a progress bar for better tracking.
- The cleaning process included:
    - Removing unnecessary characters and symbols.
    - Transliteration of Derja (e.g., `3 → Ain`, `7 → Ha`).

---

#### 5. **Removing Empty Reviews**
- After cleaning, some reviews became empty (e.g., reviews that were only emojis). These rows were removed to maintain meaningful data.

---

#### 6. **Validation of Derja Transformation**
- To validate the cleaning process, we sampled reviews containing Arabizi characters (`3` or `7`) and displayed their original and cleaned versions. This step demonstrated how the transliteration was applied.

---

#### 7. **Saving Cleaned Data**
- The cleaned dataset was saved to `../data/02_intermediate/tunisian_telco_reviews_cleaned.csv` for use in subsequent steps.

---

These preprocessing steps ensure that the data is clean, consistent, and ready for sentiment analysis and topic modeling.

In [3]:
# notebooks/03_preprocessing.ipynb

import pandas as pd
import sys
import os
from tqdm import tqdm

# Add the 'src' folder to Python's path so we can import our new script
sys.path.append(os.path.abspath('../src'))

from preprocessing import TunisianTextCleaner

# 1. Load Raw Data
RAW_DATA_PATH = "../data/01_raw/tunisian_telco_reviews_raw.csv"
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError("Run the scraping notebook first!")

df = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded {len(df)} raw reviews.")

# Drop rows where 'content' is NaN
df = df.dropna(subset=['content'])

# Removed every content with non derja digits to not confuse the model later on
non_derja_digits = r'[0146]'
df = df[~df['content'].str.contains(non_derja_digits, regex=True, na=False)]

# 2. Initialize the Cleaner
cleaner = TunisianTextCleaner()

# 3. Apply Cleaning (with Progress Bar)
# This might take 10-20 seconds
tqdm.pandas(desc="Cleaning Reviews")
df['content_cleaned'] = df['content'].progress_apply(cleaner.clean_text)

# 4. Remove empty rows
# (Some reviews are JUST emojis, which become empty strings after cleaning)
df_clean = df[df['content_cleaned'].str.len() > 2].copy()

print(f"✅ Reviews remaining after cleaning: {len(df_clean)}")

# 5. The "Genius" Validation
# Show the teacher how we handled Derja
print("\nDerja TRANSFORMATION EXAMPLES:")
print("="*50)

# Find reviews that originally had '3' or '7' and show the change
arabizi_sample = df[df['content'].str.contains(r'[37]', regex=True)].sample(5, random_state=42)

for i, row in arabizi_sample.iterrows():
    print(f"ORIGINAL: {row['content']}")
    print(f"CLEANED : {row['content_cleaned']}")
    print("-" * 50)

# 6. Save the Clean Data
OUTPUT_PATH = "../data/02_intermediate/tunisian_telco_reviews_cleaned.csv"
os.makedirs("../data/02_intermediate", exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"--> Saved clean data to: {OUTPUT_PATH}")

Loaded 23931 raw reviews.


Cleaning Reviews: 100%|██████████| 22810/22810 [00:00<00:00, 25253.99it/s]


✅ Reviews remaining after cleaning: 20607

Derja TRANSFORMATION EXAMPLES:
ORIGINAL: Sayeb l'internet lcha3b
CLEANED : sayeb linternet lchaab
--------------------------------------------------
ORIGINAL: Fokou zboubna na3in zok omkom
CLEANED : fokou zboubna naain zok omkom
--------------------------------------------------
ORIGINAL: Ya3nii khyr min blesh...
CLEANED : yaanii khyr min blesh
--------------------------------------------------
ORIGINAL: Mibonin ta7ana
CLEANED : mibonin tahana
--------------------------------------------------
ORIGINAL: ooredoo Tunisie c'est la notion du capitalisme sauvage ( jeux ereb7 3la 3ajla)
CLEANED : ooredoo tunisie cest la notion du capitalisme sauvage jeux erebh ala aajla
--------------------------------------------------
--> Saved clean data to: ../data/02_intermediate/tunisian_telco_reviews_cleaned.csv
